# 03 — Qualidade das atividades: validade

**Dimensão:** validade (ver `AGENTS.md`, seção "Qualidade de dados").

**Objetivo:** achar distâncias e durações de treino que não podem ser reais, sem confundir com o que só está vazio.

**Pedido:**
1. contar os treinos e quantos têm `distancia_km` preenchida;
2. mostrar mínimo, mediana e máximo de `distancia_km` e `duracao_min`;
3. contar os treinos acima de 200 km e os com duração zero ou negativa;
4. listar os cinco mais longos, com todas as colunas.

**Referência (roteiro do professor):** 1.254.475 treinos, 679.640 com distância; mediana 6,47 km; 951 acima de 200 km; o maior, 10.012,75 km, é um pedal de 160 minutos do apple_health; 3.649 com duração zero, nenhum negativo. Armadilha: os 574.835 sem distância não são erro.

Nada em `dados/brutos/` é alterado.

In [ ]:
import pandas as pd
from pathlib import Path

BRUTOS = Path('../dados/brutos')
atividades = pd.read_parquet(BRUTOS / 'atividades.parquet')  # engine pyarrow
atividades.dtypes

## 1. Treinos e distância preenchida

In [ ]:
total = len(atividades)
com_distancia = atividades['distancia_km'].notna().sum()
print(f'Treinos: {total:,} | com distância: {com_distancia:,} | sem distância: {total - com_distancia:,}')
assert total == 1_254_475 and com_distancia == 679_640

## 2. Mínimo, mediana e máximo

In [ ]:
resumo = atividades[['distancia_km', 'duracao_min']].agg(['min', 'median', 'max']).T
resumo

In [ ]:
assert round(resumo.loc['distancia_km', 'median'], 2) == 6.47
assert resumo.loc['distancia_km', 'max'] == 10_012.75

## 3. Acima de 200 km e duração zero ou negativa

In [ ]:
acima_200 = atividades['distancia_km'] > 200
duracao_zero = atividades['duracao_min'] == 0
duracao_negativa = atividades['duracao_min'] < 0
print(f'Acima de 200 km: {acima_200.sum():,}')
print(f'Duração zero:    {duracao_zero.sum():,}')
print(f'Duração negativa: {duracao_negativa.sum():,}')
assert acima_200.sum() == 951 and duracao_zero.sum() == 3_649 and duracao_negativa.sum() == 0

## 4. Os cinco mais longos

In [ ]:
top5 = atividades.nlargest(5, 'distancia_km')
maior = top5.iloc[0]
assert (maior['tipo'], maior['duracao_min'], maior['origem']) == ('pedal', 160, 'apple_health')
top5

## A armadilha: sem distância não é erro (quase sempre)

O roteiro diz que os 574.835 treinos sem distância "não tinham GPS". Conferindo por `tipo` e `gps_ok`, a explicação se divide em dois casos:
- **treinos em vídeo** (força, funcional, HIIT, yoga) não têm percurso: vazio esperado. Curiosamente, todos vêm com `gps_ok = True`, então `gps_ok` não significa "tinha GPS";
- **treinos ao ar livre com `gps_ok = False`**: o GPS falhou. Também não é distância inválida, mas é perda de informação.

In [ ]:
VIDEO = {'forca', 'funcional', 'hiit', 'yoga', 'strength'}
atividades['modalidade'] = atividades['tipo'].map(lambda t: 'video' if t in VIDEO else 'ar_livre')

sem_distancia = atividades[atividades['distancia_km'].isna()]
explicacao = sem_distancia.groupby(['modalidade', 'gps_ok']).size().rename('treinos')
display(explicacao)

video = explicacao.loc[('video', True)]
gps_falhou = explicacao.loc[('ar_livre', False)]
assert video + gps_falhou == len(sem_distancia) == 574_835, 'todo vazio é explicado por vídeo ou falha de GPS'
print(f'Vídeo: {video:,} | ar livre com GPS falho: {gps_falhou:,} | sem explicação: 0')

## Além do roteiro: o ritmo como detector

O corte de 200 km é arbitrário: deixa passar uma "corrida" de 150 km e não diz nada sobre treinos menores. Uma regra melhor usa as três colunas juntas. Se o treino é real, **distância ≈ duração ÷ ritmo**.

Nos treinos com duração e ritmo, a razão entre a distância informada e a implícita fica em 1,000 (±0,3%) na imensa maioria. Fora disso, a distância não é compatível com o próprio treino.

In [ ]:
avaliavel = atividades[(atividades['duracao_min'] > 0) & atividades['ritmo_min_km'].notna()].copy()
avaliavel['distancia_pelo_ritmo'] = avaliavel['duracao_min'] / avaliavel['ritmo_min_km']
avaliavel['razao'] = avaliavel['distancia_km'] / avaliavel['distancia_pelo_ritmo']
print(avaliavel['razao'].describe(percentiles=[.01, .5, .99]).round(3).to_string())

incompativel = (avaliavel['razao'] - 1).abs() > 0.10
print(f'\nDistância incompatível com duração ÷ ritmo (±10%): {incompativel.sum():,}')
print(pd.crosstab(incompativel, avaliavel['distancia_km'] > 200,
                  rownames=['incompatível'], colnames=['acima de 200 km']))

In [ ]:
# Os incompatíveis abaixo de 200 km: o que o corte fixo deixa passar
avaliavel[incompativel & (avaliavel['distancia_km'] <= 200)].sort_values('razao', ascending=False).head(10)

Nos treinos acima de 200 km, a distância informada é de 40 a 130 vezes a implícita, sem um fator único: não é erro de unidade que se converta, é valor corrompido. O ritmo informado continua plausível (1,8 a 13 min/km), por isso a distância implícita pode substituir a informada quando for preciso.

## Outros achados de validade

In [ ]:
# Categorias de tipo em dois idiomas
display(atividades['tipo'].value_counts())

# Duração zero com distância preenchida: treino de distância que não durou nada
zero_com_distancia = (duracao_zero & atividades['distancia_km'].notna()).sum()
print(f'Duração zero com distância preenchida: {zero_com_distancia:,}')

# GPS falho, mas com distância: distância sem ritmo
gps_falho_com_dist = atividades[~atividades['gps_ok'] & atividades['distancia_km'].notna()]
print(f'GPS falho com distância: {len(gps_falho_com_dist):,} | sem ritmo nesses: {gps_falho_com_dist["ritmo_min_km"].isna().sum():,}')

## Conclusão

- **Roteiro conferido:** 1.254.475 treinos, 679.640 com distância, mediana 6,47 km, 951 acima de 200 km (o maior, 10.012,75 km, um pedal de 160 min do apple_health), 3.649 com duração zero e nenhum negativo.
- **Vazio não é erro:** os 574.835 sem distância são todos explicados: treinos em vídeo (vazio esperado) ou ao ar livre com GPS falho. `gps_ok` não significa "tinha GPS": vídeo vem com `True`.
- **Detector melhor que 200 km:** distância ≈ duração ÷ ritmo. Pega os treinos acima de 200 km e também os incompatíveis abaixo disso, que o corte fixo deixa passar.
- **Decisão:** não corrigir em `dados/brutos/`. Marcar como distância inválida os incompatíveis (±10%); quando a distância for necessária, usar a implícita pelo ritmo. Duração zero fica marcada como inválida.
- **Validade de categorias:** `tipo` mistura português e inglês (`corrida`/`run`, `pedal`/`ride`, `caminhada`/`walk`, `forca`/`strength`). Unificar antes de qualquer contagem por tipo.

Decisões registradas em `decisoes.md`.